# An Introduction to Reinforcement Learning in Interactive Systems: Modeling Sequential Decision Making in HCI
By: Thomas Langerak, Aalto University, thomas.langerak@aalto.fi

Summerschool on Computational Interaction 2026, Glasgow

## Running Example: NotificationScheduler

Assume you are a phone. You receive multiple messages for the user, with different levels of importance and urgency. The user might be busy with important work. What might be useful to sense? When do you notify the users of messages? In this tutorial we will look at various ways of solving this question. From random, heuristic, and myopic decision makers; all the way to DQN. 

![notification scheduler](images/notification.png)

### The task

An agent manages a queue of incoming notifications and decides, at every timestep, whether to:
- **wait** — protect the user's focus, but notifications lose value over time, or  
- **deliver** — interrupt the user and flush the entire queue.

### The POMDP structure

The user's *concentration level* (low / medium / high) is **hidden** — the agent cannot observe it directly. Interrupting a highly-focused user is costly; interrupting an idle one is cheap. The agent must infer concentration from the observable context.

```
Hidden:     concentration
Observable: context, time_of_day, time_since_delivery,
            queue_size, max_urgency, max_importance
```

## Setup

In [ ]:
import sys, os
REPO_ROOT = os.path.dirname(os.path.abspath("demo.ipynb"))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
from stable_baselines3 import DQN
import tqdm
import numpy as np
%matplotlib inline

import notification_env  # registers NotificationScheduler-v0
from notification_env import NotificationSchedulerEnv
from notification_env.notification_scheduler import (
    CONTEXT_LABELS, TOD_LABELS, CONCENTRATION_LABELS,
)
from notification_env.utils import run_episode_with_frames
from notification_env.plotting import animate

env = NotificationSchedulerEnv()
obs, info = env.reset(seed=0)
print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print("Sample obs:       ", obs)
env.close()

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Ex 1: Exploring the notification environment**

Run the cell below and inspect the observation vector. What does each field represent? Which value is hidden from the agent?

</div>

In [ ]:
env = NotificationSchedulerEnv()
obs, _ = env.reset(seed=42)

obs_labels = [
    "context          (0=idle, 1=focus, 2=meeting)",
    "time_of_day      (0=morning, 1=afternoon, 2=evening)",
    "time_since_delivery  (0=just, 1=short, 2=med, 3=long)",
    "queue_size       (0-8)",
    "max_urgency      (0=none, 1=low, 2=med, 3=high)",
    "max_importance   (0=none, 1=low, 2=med, 3=high)",
]
print("=== Observable ===")
for label, val in zip(obs_labels, obs):
    print(f"  {label:52s} = {val}")
print()
print("=== Hidden (not available to the agent) ===")
print(f"  concentration = {env.concentration}  ({CONCENTRATION_LABELS[env.concentration]})")
print("=== Action space ===")
print(f"  0 = wait, 1 = deliver  (action_space: {env.action_space})")


## Why is notification scheduling hard?

These naive approaches each have fundamental flaws:

| Method | Issue |
|--------|-------|
| Random | Never really optimal |
| Heuristic | Writing comprehensive rules is hard and error-prone |
| Myopic | An immediate payout is not always ideal |

This is a **sequential decision making** problem:

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

**Challenges**

- **Sequential**: Each action changes the world and all subsequent decisions.
- **Delayed rewards**: The high pay-off may only come after many correct decisions.
- **Stochasticity**: Same decision in the same state can lead to different outcomes.
- **Partial observability**: You cannot observe everything (e.g., inside a user's head).

*Sequential decision making is about making the right choice, given the information you have, to maximise a future payout.*

</div>

Sequential decision making is ubiquitous in HCI:

| Domain | State | Action | Goal |
|--------|-------|--------|------|
| Autocorrect | Interaction history | Correct / ignore | Minimise errors |
| Adaptive menus | Usage history | Item assignment | Maximise performance |
| Virtual assistant | User context | When to intervene | Minimise disruption |

## Markov Decision Processes

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

A **Markov Decision Process (MDP)** is the standard formal framework for sequential decision making under uncertainty.

$$\{S, A, T, R, \gamma\}$$

- **States** $S$ — a representation of the environment (e.g., pixels, location, queue contents)
- **Actions** $A$ — the choices an agent can take
- **Transition** $T(s' \mid s, a)$ — probability of reaching state $s'$ after taking action $a$ in state $s$
- **Reward** $R(s, a)$ — the utility of taking action $a$ in state $s$
- **Discount factor** $\gamma \in [0, 1)$ — how much we value future rewards vs. immediate ones

**Markov property**: the future depends only on the current state and action, not on the full history.

</div>

When part of the state is unobservable (like concentration), we have a **Partially Observable MDP (POMDP)**:

$$\{S, A, T, R, \gamma, \Omega, O\}$$

where $\Omega$ is the set of possible observations and $O(o \mid s)$ is the probability of making observation $o$ in state $s$.

**MDP vs. RL**: An MDP is a *prescriptive model* — a full description of the world. Reinforcement learning is a framework for how an agent can *learn* optimal behaviour in an unknown or partially known MDP.

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Ex 2a: Random agent**

Implement an agent that returns a random action (0 = wait, 1 = deliver) with equal probability. Random is always a good first baseline.

</div>

In [ ]:
rng = np.random.default_rng(7)

def random_policy(obs):
    # TODO: return a random action (0=wait or 1=deliver)
    raise NotImplementedError

try:
    rand_reward, rand_frames, env_rand = run_episode_with_frames(random_policy, seed=0)
    print(f"Total reward : {rand_reward:.2f}")
    print(f"Deliveries   : {sum(env_rand.action_history)}")
except NotImplementedError:
    print("⚠️  Not implemented yet — fill in random_policy first.")

In [ ]:
# Solution
rng = np.random.default_rng(7)

def random_policy(obs):
    return int(rng.integers(0, 2))

rand_reward, rand_frames, env_rand = run_episode_with_frames(random_policy, seed=0)
print(f"Total reward : {rand_reward:.2f}")
print(f"Deliveries   : {sum(env_rand.action_history)}")

The result is shown numerically above — typically around 24–25. Below is a visualisation of a full episode. The figure has five panels:

- **Notification Queue** — each bar is one notification. Height = current value (decays over time), colour = urgency (green=low, orange=medium, red=high).
- **User State** — context (idle / focus / meeting), time of day, and hidden concentration (educational only — agent cannot see this).
- **Cumulative Reward** — total reward so far.
- **Action History** — 0 = wait, 1 = deliver. Each spike is a delivery.
- **Concentration Over Time** — red dashed lines mark delivery events, which cause a concentration drop.

In [ ]:
animate(rand_frames)

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Ex 2b: Heuristic agent**

Implement a hand-coded rule. For example: deliver if there are 3+ notifications, or if max urgency is high, or every N steps. Keep it simple.

The `obs` vector:

| Index | Name | Values |
|-------|------|--------|
| `obs[0]` | context | 0 = idle, 1 = focus, 2 = meeting |
| `obs[1]` | time of day | 0 = morning, 1 = afternoon, 2 = evening |
| `obs[2]` | time since delivery | 0 = just, 1 = short, 2 = medium, 3 = long |
| `obs[3]` | queue size | 0–8 |
| `obs[4]` | max urgency | 0 = none, 1 = low, 2 = medium, 3 = high |
| `obs[5]` | max importance | 0 = none, 1 = low, 2 = medium, 3 = high |

</div>

In [ ]:
def heuristic_policy(obs):
    # TODO: implement a heuristic policy of choice, e.g., return 1 (deliver) if the queue has 3+ notifications, or max urgency is high, or at fixed intervals, etc. 
    # otherwise return 0 (wait).
    raise NotImplementedError

try:
    heur_reward, heur_frames, env_heur = run_episode_with_frames(heuristic_policy, seed=0)
    print(f"Total reward : {heur_reward:.2f}")
    print(f"Deliveries   : {sum(env_heur.action_history)}")
except NotImplementedError:
    print("⚠️  Not implemented yet — fill in heuristic_policy first.")

In [ ]:
# Solution
def heuristic_policy(obs):
    return 1 if (obs[3] >= 3 or obs[4] == 3) else 0

heur_reward, heur_frames, env_heur = run_episode_with_frames(heuristic_policy, seed=0)
print(f"Total reward : {heur_reward:.2f}")
print(f"Deliveries   : {sum(env_heur.action_history)}")

In [ ]:
animate(heur_frames)

## Ex 2c: Myopic agent

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

Instead of a hand-coded rule, we compute the **immediate reward** of each action and pick the best:

$$\pi(s) = \arg\max_a \; R(s, a)$$

This is called **myopic** — it only looks at the reward right now, ignoring all future consequences.

The reward depends on concentration $c$ — which is hidden. The agent uses $\mathbb{E}[c \mid \text{context}]$: the expected concentration given the observable context, derived from the stationary distribution of the concentration Markov chain.

**`env.simulate(action)`** returns the reward that *would* result from taking that action, without stepping the environment. Internally it uses $\mathbb{E}[c \mid \text{context}]$, so no hidden state is read.

</div>

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Exercise 2c**

Use `env_myopic.simulate(action)` for both actions and return the one with the higher reward.

</div>

In [ ]:
env_myopic = NotificationSchedulerEnv(render_mode="rgb_array")

def myopic_policy(obs):
    # TODO: use env_myopic.simulate(action) to get the immediate reward
    # for each action, then return the action with the higher reward.
    raise NotImplementedError

try:
    myop_reward, myop_frames, env_myopic = run_episode_with_frames(
        myopic_policy, seed=0, env=env_myopic
    )
    print(f"Total reward : {myop_reward:.2f}")
    print(f"Deliveries   : {sum(env_myopic.action_history)}")
except NotImplementedError:
    print("⚠️  Not implemented yet — fill in myopic_policy first.")

In [ ]:
# Solution
env_myopic = NotificationSchedulerEnv(render_mode="rgb_array")

def myopic_policy(obs):
    r_wait,    _ = env_myopic.simulate(0)
    r_deliver, _ = env_myopic.simulate(1)
    return 1 if r_deliver > r_wait else 0

myop_reward, myop_frames, env_myopic = run_episode_with_frames(
    myopic_policy, seed=0, env=env_myopic
)
print(f"Total reward : {myop_reward:.2f}")
print(f"Deliveries   : {sum(env_myopic.action_history)}")

In [ ]:
animate(myop_frames)

## From Myopic to Optimal: Value Functions

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

The myopic agent only looks at immediate reward. The right objective is the **total discounted return**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

We describe our strategy with a **policy** $\pi: S \rightarrow A$. The **value function** $V^\pi(s)$ is the expected total return from state $s$ following $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$$

This satisfies the **Bellman equation** — a recursive identity:

$$V^\pi(s) = \mathbb{E}_\pi\left[R_{t+1} + \gamma V^\pi(S_{t+1}) \mid S_t = s\right]$$

The **optimal value function** $V^*(s)$ satisfies the **Bellman optimality equation**:

$$V^*(s) = \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V^*(s') \right]$$

The optimal policy follows immediately:

$$\pi^*(s) = \arg\max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s') \right]$$

When $\gamma = 0$ this reduces to the myopic agent. Increasing $\gamma$ makes the agent plan further ahead.

</div>

## Value Iteration: Dynamic Programming

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

**Value iteration** solves the Bellman optimality equation by starting with $V(s) = 0$ everywhere and repeatedly applying the update until $V$ converges:

$$V_{k+1}(s) = \max_a \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) \, V_k(s') \right]$$

```
initialise V(s) = 0  for all states s
repeat:
    for each state s:
        V(s) <- max_a [ R(s,a) + gamma * sum_{s'} P(s'|s,a) * V(s') ]
until max|delta V| < threshold
policy(s) = argmax_a [ R(s,a) + gamma * sum_{s'} P(s'|s,a) * V(s') ]
```

**Guarantees**: Converges in a finite number of iterations to the optimal policy — given the full MDP.

**Requirement**: You need the full transition model $P(s'|s,a)$ and reward function $R(s,a)$. This is the *model-based* assumption.

</div>

## Ex 5: Value iteration (implementation)

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

**POMDP approximations**

To run value iteration on our notification problem we need $R(s,a)$ and $P(s'|s,a)$ for every observable state $s$. Because concentration is hidden we approximate:

1. **R(s, a)** — replace concentration with $\mathbb{E}[c \mid \text{context}]$, from the stationary distribution of the concentration Markov chain.
2. **Notification value** — estimated from `queue_size` and `max_urgency` using average ages and importances.

The observable state space is 3 x 3 x 4 x 9 x 4 x 4 = **5,184 states** — small enough to enumerate.

</div>

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Exercise 5**

Implement the Bellman update inside `run_value_iteration()`. Replace `val = None` with:

```python
val = R[s, a] + gamma * sum(p * V[s_next] for p, s_next in T[s][a])
```

- `R[s, a]` — immediate reward
- `T[s][a]` — list of `(probability, next_state)` pairs
- `V[s_next]` — current value estimate of the next state

**Bonus**: Set `gamma = 0.0`. What policy do you get, and why?

</div>

In [ ]:
from notification_env.value_iteration import (
    build_reward_model,
    build_transition_model,
    ValueIterationAgent,
    n_states, N_ACTIONS,
    state_to_index,
)
from notification_env.plotting import plot_convergence, plot_value_function

_cfg_env = NotificationSchedulerEnv()
N_STATES = n_states(_cfg_env)

In [ ]:
print("Building reward model R(s, a) ...")
R = build_reward_model(_cfg_env)
print(f"  shape={R.shape}  range=[{R.min():.2f}, {R.max():.2f}]")

print("Building transition model P(s'|s, a) ...")
T = build_transition_model(_cfg_env)
print(f"  {N_STATES} states × {N_ACTIONS} actions")

In [ ]:
gamma    = 0.95   # discount factor — try 0.0, 0.5, 0.99
theta    = 1e-6   # convergence threshold
max_iter = 10_000

def run_value_iteration():
    V      = np.zeros(N_STATES)
    deltas = []

    for it in range(max_iter):
        V_new = np.zeros(N_STATES)

        for s in range(N_STATES):
            best = -np.inf
            for a in range(2):  # 0=wait, 1=deliver
                # TODO: implement the Bellman optimality equation:
                #
                #   V(s) ← max_a [ R(s,a) + γ · Σ_{s'} P(s'|s,a) · V(s') ]
                #
                # Hint: R[s, a] gives the immediate reward.
                #       T[s][a] is a list of (prob, next_state) pairs.
                #       V[s_next] is the current value estimate of next_state.
                val = None  # replace with your implementation
                raise NotImplementedError
                if val > best:
                    best = val
            V_new[s] = best

        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new

        if delta < theta:
            print(f"Converged in {it + 1} iterations (delta={delta:.2e})")
            break

    # Policy: for each state, pick the action with the highest Q-value
    Q = np.array([
        [R[s, a] + gamma * sum(p * V[s_next] for p, s_next in T[s][a]) for a in range(2)]
        for s in range(N_STATES)
    ])
    policy = np.argmax(Q, axis=1)

    print(f"V range : [{V.min():.2f}, {V.max():.2f}]")
    print(f"Policy  : {int((policy==0).sum())} wait states, {int((policy==1).sum())} deliver states")
    return V, policy, deltas

try:
    V, policy, deltas = run_value_iteration()
except NotImplementedError:
    print("⚠️  Not implemented yet — fill in the Bellman update (val = ...) first.")

In [ ]:
# Solution
gamma    = 0.95
theta    = 1e-6
max_iter = 10_000

V      = np.zeros(N_STATES)
deltas = []

for it in range(max_iter):
    V_new = np.zeros(N_STATES)

    for s in range(N_STATES):
        best = -np.inf
        for a in range(2):
            future = sum(p * V[s_next] for p, s_next in T[s][a])
            val = R[s, a] + gamma * future
            if val > best:
                best = val
        V_new[s] = best

    delta = np.max(np.abs(V_new - V))
    deltas.append(delta)
    V = V_new

    if delta < theta:
        print(f"Converged in {it + 1} iterations (delta={delta:.2e})")
        break

# Policy: for each state, pick the action with the highest Q-value
Q = np.array([
    [R[s, a] + gamma * sum(p * V[s_next] for p, s_next in T[s][a]) for a in range(2)]
    for s in range(N_STATES)
])
policy = np.argmax(Q, axis=1)

print(f"V range : [{V.min():.2f}, {V.max():.2f}]")
print(f"Policy  : {int((policy==0).sum())} wait states, {int((policy==1).sum())} deliver states")

In [ ]:
plot_convergence(deltas, title="Value iteration convergence", ylabel="Max |ΔV|")

In [ ]:
# V(s) vs queue size, sliced at focus / morning / long since delivery
plot_value_function(V, lambda s: state_to_index(s, _cfg_env))

**Bonus Question**: Why does the value drop for bigger queues?

<details><summary>Hint</summary>
A bigger queue means the agent is likely to deliver soon regardless. Delivering a large queue to a concentrated user is expensive. Large queues also result from long waits, during which notification value has already decayed.
</details>

In [ ]:
vi_agent = ValueIterationAgent(Q, _cfg_env)
vi_reward, vi_frames, env_vi = run_episode_with_frames(vi_agent, seed=0)
print(f"Total reward : {vi_reward:.2f}")
print(f"Deliveries   : {sum(env_vi.action_history)}")

In [ ]:
animate(vi_frames)

## Pros and Cons of Value Iteration

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

**Pros**
- Guaranteed to converge to the optimal policy in a finite number of iterations
- Principled: grounded in the Bellman optimality equations

**Cons**
- Requires the full transition model $P(s'|s,a)$ — you must know the environment dynamics
- Must iterate over the entire state space — intractable for large problems (Go has $10^{170}$ states)
- We had to *approximate* both $R$ and $P$ due to the hidden concentration

These limitations motivate moving from model-based planning to model-free reinforcement learning.

</div>

## Ex 6: Q-learning

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

### Why value iteration breaks down

Value iteration required two hand-crafted models: a reward model $R(s,a)$ and a transition model $P(s'|s,a)$. In realistic settings — a different user, richer observations, continuous states — writing down $P$ and $R$ becomes intractable.

### Q-learning: model-free RL

**Q-learning** sidesteps both by sampling real transitions from the environment. It learns $Q^*(s, a)$ — the value of taking action $a$ in state $s$:

$$Q^*(s, a) = R(s, a) + \gamma \sum_{s'} P(s'|s,a)\, \max_{a'} Q^*(s', a')$$

Once $Q$ is known: $\pi(s) = \arg\max_a Q(s, a)$ — no model needed.

### The TD update

After observing transition $(s, a, r, s')$:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} - Q(s, a) \right]$$

- $\alpha$ — learning rate
- TD target — what the Bellman equation says $Q(s,a)$ should be
- TD target $-$ $Q(s,a)$ — **TD error**: how wrong we are

### Exploration: epsilon-greedy

With probability $\varepsilon$: pick a random action (explore).
With probability $1-\varepsilon$: pick $\arg\max_a Q(s,a)$ (exploit).
$\varepsilon$ is decayed over training.

</div>

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Exercise 6**

Implement the Q-learning TD update inside `run_q_learning()`.

</div>

In [ ]:
alpha      = 0.1     # learning rate
gamma      = 0.95    # discount factor
eps        = 1.0     # starting exploration rate
eps_min    = 0.05    # minimum exploration rate
eps_decay  = 0.9995  # multiplicative decay per episode
n_episodes = 100_000

def run_q_learning():
    Q = np.zeros((N_STATES, N_ACTIONS))
    td_errors = []
    _eps = eps

    for ep in range(n_episodes):
        env_ql = NotificationSchedulerEnv()
        obs, _ = env_ql.reset(seed=ep)
        ep_errors = []
        done = False

        while not done:
            s = state_to_index(tuple(int(x) for x in obs), _cfg_env)

            # epsilon-greedy action selection
            if np.random.random() < _eps:
                a = env_ql.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))

            obs_next, r, terminated, truncated, _ = env_ql.step(a)
            done = terminated or truncated
            s_next = state_to_index(tuple(int(x) for x in obs_next), _cfg_env)

            # TODO: implement the Q-learning update:
            #
            #   Q(s, a) <- Q(s, a) + alpha * [ r + gamma * max_a' Q(s', a') - Q(s, a) ]
            #
            # Hint: np.max(Q[s_next]) gives max_a' Q(s', a')
            raise NotImplementedError

            obs = obs_next

        td_errors.append(np.mean(ep_errors) if ep_errors else 0.0)
        _eps = max(eps_min, _eps * eps_decay)

    print(f"Final eps : {_eps:.3f}")
    print(f"Q range : [{Q.min():.2f}, {Q.max():.2f}]")
    return Q, td_errors

try:
    Q, td_errors = run_q_learning()
except NotImplementedError:
    print("⚠️  Not implemented yet — fill in the Q-learning update first.")

In [ ]:
# Solution
alpha      = 0.1
gamma      = 0.95
eps        = 1.0
eps_min    = 0.05
eps_decay  = 0.9995
n_episodes = 10_000

Q = np.zeros((N_STATES, N_ACTIONS))
td_errors = []

for ep in range(n_episodes):
    env_ql = NotificationSchedulerEnv()
    obs, _ = env_ql.reset(seed=ep)
    ep_errors = []
    done = False

    while not done:
        s = state_to_index(tuple(int(x) for x in obs), _cfg_env)

        if np.random.random() < eps:
            a = env_ql.action_space.sample()
        else:
            a = int(np.argmax(Q[s]))

        obs_next, r, terminated, truncated, _ = env_ql.step(a)
        done = terminated or truncated
        s_next = state_to_index(tuple(int(x) for x in obs_next), _cfg_env)

        td_target = r + gamma * np.max(Q[s_next])
        td_error  = td_target - Q[s, a]
        Q[s, a]  += alpha * td_error

        ep_errors.append(abs(td_error))
        obs = obs_next

    td_errors.append(np.mean(ep_errors) if ep_errors else 0.0)
    eps = max(eps_min, eps * eps_decay)

print(f"Final eps : {eps:.3f}")
print(f"Q range : [{Q.min():.2f}, {Q.max():.2f}]")

In [ ]:
def ql_policy(obs):
    s = state_to_index(tuple(int(x) for x in obs), _cfg_env)
    return int(np.argmax(Q[s]))

ql_reward, ql_frames, env_ql_eval = run_episode_with_frames(ql_policy, seed=0)
print(f"Total reward : {ql_reward:.2f}")
print(f"Deliveries   : {sum(env_ql_eval.action_history)}")

In [ ]:
animate(ql_frames)

### Convergence and policy

In [ ]:
window = 200
smoothed = np.convolve(td_errors, np.ones(window)/window, mode='valid')
plot_convergence(smoothed, title=f"Q-learning TD error ({window}-episode rolling avg)", ylabel="Mean |TD error|")

## Pros and Cons of Q-Learning

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

**Pros**
- No need to know system dynamics — learns from real experience
- Converges to the optimal policy (under sufficient exploration)
- Learns the optimal Q-function even while following an exploratory policy

**Cons**
- High variance — individual transitions are noisy estimates
- **Tabular**: one entry per state-action pair — doesn't scale to large or continuous spaces
- States never visited during training have Q = 0 (default policy)

Our Q-learning agent visited only ~600 of 5,184 states. This **coverage problem** motivates moving to function approximation.

</div>

## Ex 7: DQN (Deep Q-Network)

<div style="background:#f5f5f5; padding:14px 16px; border-radius:6px; border-left:5px solid #9e9e9e; margin:12px 0;">

### From tabular to deep

Tabular Q-learning stores one value per $(s, a)$ pair. It has two hard limits:

1. **Coverage** — states not visited during training have Q = 0.
2. **No generalisation** — nothing is shared between similar states.

Real HCI problems are worse: raw images, continuous motor states, rich UI layouts can have astronomically many states.

**DQN** replaces the table with a neural network: $Q(s, a; \theta)$. The network *generalises* — it can predict Q-values for states it has never seen. Everything else stays the same: same Bellman equation, same TD update, same $\varepsilon$-greedy exploration. The update becomes gradient descent on:

$$\mathcal{L}(\theta) = \left( r + \gamma \max_{a'} Q_\theta(s', a') - Q_\theta(s, a) \right)^2$$

### Stable Baselines 3

Rather than implementing DQN from scratch, we use [Stable Baselines 3](https://stable-baselines3.readthedocs.io/). The environment is already a standard Gymnasium env, so it plugs straight in.

</div>

<div style="background:#ffcccc; padding:14px 16px; border-radius:6px; border-left:5px solid #cc0000; margin:12px 0;">

**Exercise 7**

Run the DQN training below. Then answer:

1. Does DQN outperform tabular Q-learning? Why or why not?
2. Does DQN match value iteration? What are the remaining limitations?
3. How many timesteps did it take? How does that compare to the number of states?

</div>

In [ ]:
from stable_baselines3 import DQN

dqn_model = DQN(
    "MlpPolicy",
    NotificationSchedulerEnv(),
    learning_rate  = 1e-3,
    batch_size     = 64,
    gamma          = 0.95,
    exploration_fraction   = 0.4,   # fraction of training spent decaying eps
    exploration_final_eps  = 0.05,
    verbose = 0,
)
dqn_model.learn(total_timesteps=250_000, progress_bar=True)
print("Training done.")

In [ ]:
def dqn_policy(obs):
    action, _ = dqn_model.predict(obs, deterministic=True)
    return int(action)

dqn_reward, dqn_frames, env_dqn = run_episode_with_frames(dqn_policy, seed=0)
print(f"Total reward : {dqn_reward:.2f}")
print(f"Deliveries   : {sum(env_dqn.action_history)}")

In [ ]:
animate(dqn_frames)